# télos MDLM: Overnight Training Suites
Run these cells to execute sequential training sweeps. These suites handle everything: environment setup, sequential `train_mlx.py` or `train.py` execution, logging, and auto-evaluation.

## 50M Parameter Ratio Study (1:1, 1:10, 1:20)
Executes 3 training runs back-to-back on Apple MLX with micro-batch size 32 and gradient accumulation 8.
Logs results to `logs/overnight_50m_ratio_study.log`.

In [ ]:
import time
import subprocess
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
LOG_FILE = Path("logs/overnight_50m_ratio_study.log")

SUITE = [
    ("1:1 Ratio (50M Tokens)", "configs/phase_b_50m_1to1_mlx.yaml", "checkpoints/phase_b_50m_1to1_mlx"),
    ("1:10 Ratio (500M Tokens)", "configs/phase_b_50m_1to10_mlx.yaml", "checkpoints/phase_b_50m_1to10_mlx"),
]

def log(msg: str):
    timestamp = time.strftime("[%Y-%m-%d %H:%M:%S]")
    line = f"{timestamp} {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

log("==========================================================================")
log(" STARTING OVERNIGHT 50M PARAMETER RATIO STUDY")
log("==========================================================================")

suite_start = time.time()

for idx, (label, config_path, ckpt_dir) in enumerate(SUITE, start=1):
    log(f"\n[{idx}/{len(SUITE)}] STARTING TRAINING: {label}")
    run_start = time.time()

    cmd_train = ["uv", "run", "python", "scripts/train_mlx.py", "--config", config_path]
    proc = subprocess.run(cmd_train, capture_output=False)

    if proc.returncode != 0:
        log(f"ERROR: Training failed for {label}")
        continue

    log(f"SUCCESS: Completed training {label} in {(time.time() - run_start) / 60.0:.2f} minutes.")

    base_path = Path(ckpt_dir)
    matching_dirs = sorted(list(base_path.parent.glob(f"{base_path.name}*")), reverse=True)
    target_ckpt = matching_dirs[0] if matching_dirs else base_path

    log(f"Running 100 Contextual Probes for {label} on {target_ckpt}...")
    cmd_probe = ["uv", "run", "python", "scripts/sample.py", "--checkpoint", str(target_ckpt), "--mode", "probes"]
    probe_res = subprocess.run(cmd_probe, capture_output=True, text=True)

    if probe_res.returncode == 0:
        log(f"\n--- Probe Results for {label} ---\n" + probe_res.stdout)

log(f"\nOVERNIGHT 50M RATIO STUDY COMPLETED IN {(time.time() - suite_start) / 3600.0:.2f} HOURS!")


## 25M Parameter Ratio Study (1:3, 1:10, 1:15, 1:20)
Executes 4 training runs back-to-back on Apple MLX.

In [ ]:
import time
import subprocess
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
LOG_FILE = Path("logs/overnight_25m_ratio_study.log")

SUITE = [
    ("1:3 Ratio (75M Tokens)", "configs/phase_b_25m_1to3_mlx.yaml", "checkpoints/phase_b_25m_1to3_mlx"),
    ("1:10 Ratio (250M Tokens)", "configs/phase_b_25m_1to10_mlx.yaml", "checkpoints/phase_b_25m_1to10_mlx"),
    ("1:15 Ratio (375M Tokens)", "configs/phase_b_25m_1to15_mlx.yaml", "checkpoints/phase_b_25m_1to15_mlx"),
    ("1:20 Ratio (500M Tokens)", "configs/phase_b_25m_1to20_mlx.yaml", "checkpoints/phase_b_25m_1to20_mlx"),
]

def log(msg: str):
    timestamp = time.strftime("[%Y-%m-%d %H:%M:%S]")
    line = f"{timestamp} {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

log("==========================================================================")
log(" STARTING OVERNIGHT 25M PARAMETER RATIO STUDY")
log("==========================================================================")

suite_start = time.time()

for idx, (label, config_path, ckpt_dir) in enumerate(SUITE, start=1):
    log(f"\n[{idx}/{len(SUITE)}] STARTING TRAINING: {label}")
    run_start = time.time()

    cmd_train = ["uv", "run", "python", "scripts/train_mlx.py", "--config", config_path]
    proc = subprocess.run(cmd_train, capture_output=False)

    if proc.returncode != 0:
        log(f"ERROR: Training failed for {label}")
        continue

    log(f"SUCCESS: Completed training {label} in {(time.time() - run_start) / 60.0:.2f} minutes.")

    log(f"Running 100 Contextual Probes for {label}...")
    cmd_probe = ["uv", "run", "python", "scripts/sample.py", "--checkpoint", ckpt_dir, "--mode", "probes"]
    probe_res = subprocess.run(cmd_probe, capture_output=True, text=True)

    if probe_res.returncode == 0:
        log(f"\n--- Probe Results for {label} ---\n" + probe_res.stdout)

log(f"\nOVERNIGHT 25M RATIO STUDY COMPLETED IN {(time.time() - suite_start) / 3600.0:.2f} HOURS!")


## TPU v6e-1 50M Suite & HuggingFace Upload
PyTorch-XLA device auto-verification and auto-upload to HuggingFace Hub.

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path

Path("logs").mkdir(exist_ok=True)
LOG_FILE = Path("logs/tpu_50m_ratio_study.log")
HF_WRITE_TOKEN = os.environ.get("HF_TOKEN", "")
HF_REPO_ID = "Kazenowoko/telos-50m-ratio-study"

SUITE = [
    ("1:30 Ratio (1.5B Tokens)", "configs/phase_b_50m_1to30_tpu.yaml", "checkpoints/phase_b_50m_1to30_tpu"),
    ("1:40 Ratio (2.0B Tokens)", "configs/phase_b_50m_1to40_tpu.yaml", "checkpoints/phase_b_50m_1to40_tpu"),
    ("1:20 Ratio (1.0B Tokens)", "configs/phase_b_50m_1to20_tpu.yaml", "checkpoints/phase_b_50m_1to20_tpu"),
]

def log(msg: str):
    timestamp = time.strftime("[%Y-%m-%d %H:%M:%S]")
    line = f"{timestamp} {msg}"
    print(line, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(line + "\n")

def upload_to_huggingface(ckpt_dir: str, label: str):
    log(f"Uploading {label} to HuggingFace Hub ({HF_REPO_ID})...")
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(repo_id=HF_REPO_ID, repo_type="model", token=HF_WRITE_TOKEN, exist_ok=True, private=False)
        
        folder_path = Path(ckpt_dir)
        if folder_path.exists():
            api.upload_folder(
                folder_path=str(folder_path),
                path_in_repo=folder_path.name,
                repo_id=HF_REPO_ID,
                token=HF_WRITE_TOKEN
            )
            log(f"SUCCESS: Uploaded {label}")
    except Exception as e:
        log(f"Warning: HuggingFace upload failed: {e}")

env = dict(os.environ)
env["PJRT_DEVICE"] = "TPU"
env["TPU_PROCESS_BOUNDS"] = "1,1,1"
env["TPU_VISIBLE_DEVICES"] = "0"
env["PYTHONUNBUFFERED"] = "1"

log("==========================================================================")
log(" STARTING TPU v6e-1 50M PARAMETER RATIO STUDY")
log("==========================================================================")

suite_start = time.time()

for idx, (label, config_path, ckpt_dir) in enumerate(SUITE, start=1):
    log(f"\n[{idx}/{len(SUITE)}] STARTING TRAINING ON TPU: {label}")
    run_start = time.time()

    cmd_train = [sys.executable, "scripts/train.py", "--config", config_path, "--device", "tpu", "--model-size", "50M"]
    proc = subprocess.run(cmd_train, env=env, capture_output=False)

    if proc.returncode != 0:
        log(f"ERROR: TPU Training failed for {label}")
        continue

    log(f"SUCCESS: Completed TPU training in {(time.time() - run_start) / 60.0:.2f} minutes.")
    upload_to_huggingface(ckpt_dir, label)

log(f"\nTPU RATIO STUDY COMPLETED IN {(time.time() - suite_start) / 60.0:.2f} MINUTES!")
